### Example based on the official PuLP documentation

> "In this case study a wedding planner must determine guest seating allocations for a wedding. 
To model this problem the tables are modelled as the partitions and the guests invited to the wedding are modelled as the elements of S. 
The wedding planner wishes to maximise the total happiness of all of the tables.
A set partitioning problem may be modelled by explicitly enumerating each possible subset. 
Though this approach does become intractable for large numbers of items (without using column generation) it does 
have the advantage that the objective function co-efficients for the partitions can be non-linear expressions (like happiness) and still 
allow this problem to be solved using Linear Programming."

> - *PuLP Documentation: Case Studies*, https://coin-or.github.io/pulp/CaseStudies/a_set_partitioning_problem.html

The following code reproduces the provided example with minimal adaptations.


In [2]:
import pulp as pl
from typing import Tuple, Union

max_tables = 5
max_table_size = 4
guests = "A B C D E F G I J K L M N O P Q R".split()

possible_tables = [tuple(c) for c in pl.allcombinations(guests, max_table_size)]

x = pl.LpVariable.dicts("table", possible_tables, 0, 1, pl.LpInteger)

def happiness(
        table: Union[
            Tuple[str, str], Tuple[str, str, str, str], Tuple[str], Tuple[str, str, str]
        ]
) -> int:
    return abs(ord(table[0]) - ord(table[-1]))

seating_model = pl.LpProblem("Wedding Seating Model", pl.LpMinimize)

/home/lcs/optmization_problems/env/lib/python3.10/site-packages/pulp/pulp.py:1455: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


In [ ]:
seating_model += pl.lpSum(happiness(table) * x[table] for table in possible_tables)

seating_model += (pl.lpSum(x[table] for table in possible_tables) <= max_tables, "Maximum number of tables")

In [4]:
for guest in guests:
    seating_model += (pl.lpSum(x[table] for table in possible_tables if guest in table) == 1, f"Must seat {guest}")

seating_model.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/lcs/optmization_problems/env/lib/python3.10/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/43642a219acb43dabe62c30edf0d20ff-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/43642a219acb43dabe62c30edf0d20ff-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 23 COLUMNS
At line 24708 RHS
At line 24727 BOUNDS
At line 27941 ENDATA
Problem MODEL has 18 rows, 3213 columns and 15062 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 12 - 0.00 seconds
Cgl0004I processed model has 18 rows, 3213 columns (3213 integer (3213 of which binary)) and 15062 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of 12
Cbc0038I Before mini branch and bound, 3213 integers at bound fixed and 0 con

1

In [5]:
print(f"The chosen tables are out of a total of {len(possible_tables)}:")
for table in possible_tables:
    if x[table].value() == 1.0:
        print(table)

The chosen tables are out of a total of 3213:
('M', 'N')
('E', 'F', 'G')
('A', 'B', 'C', 'D')
('I', 'J', 'K', 'L')
('O', 'P', 'Q', 'R')
